# 🎬 Mini-projet : Assistant d'analyse des sentiments avec optimisation de BERT

## Scénario métier
L'équipe d'analyse du support souhaite disposer d'un **signal fiable de ressenti** pour les commentaires longs afin de pouvoir remonter les problèmes des clients mécontents avant qu'ils ne se désabonnent.

## Prérequis
- Python 3.9+
- Environnement GPU (Colab T4, Kaggle ou GPU local avec ~6 Go de VRAM)
- Packages : `tensorflow`, `tensorflow-datasets`, `transformers`, `accelerate`, `evaluate`

## Méthodologie
```
Données IMDB  →  Tokenisation BERT  →  Fine-tuning  →  Évaluation  →  Inférence
```

---
## 📦 Section 1 — Installation des dépendances

Nous réutilisons exactement la même chaîne d'outils que les jours 3 et 4 pour assurer la continuité et nous concentrer sur le nouveau flux de travail plutôt que sur de nouvelles bibliothèques.

In [ ]:
# À exécuter une seule fois dans un environnement vierge
!pip install -q tensorflow tensorflow-datasets transformers accelerate evaluate

---
## 🔍 Section 2 — Vérification des importations et du matériel

Nous commençons toujours par vérifier les versions et le matériel.  
⚠️ Si vous voyez `GPU devices: []`, changez votre environnement d'exécution pour activer le GPU (dans Colab : *Exécution → Modifier le type d'exécution → GPU T4*).

In [ ]:
import platform
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices('GPU'))

---
## 📂 Section 3 — Chargement de l'ensemble de données IMDB

Nous utilisons **IMDb** car :
- Données équilibrées : 25 000 avis positifs / 25 000 avis négatifs
- Déjà segmenté en train / test
- Benchmark classique pour l'analyse de sentiments

> 💬 `as_supervised=True` produit des paires `(text, label)`, exactement ce que notre modèle attend.

In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True
)
print(ds_info)

In [ ]:
# Aperçu rapide de quelques exemples
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")

---
## 🔤 Section 4 — Configuration du tokenizer et du pipeline de données

BERT utilise la **tokenisation WordPiece** qui :
- Décompose les mots rares en sous-mots pour une couverture complète
- Ajoute des tokens spéciaux `[CLS]` et `[SEP]` pour marquer les limites de phrases
- Utilise des **masques d'attention** pour ignorer les tokens de padding

> 🧠 Nous importons ce tokenizer pour réutiliser le même vocabulaire appris par le modèle de base en 2018.

In [ ]:
MAX_LENGTH = 256   # Tronquer ou padder chaque avis à 256 tokens
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer chargé :", tokenizer.name_or_path)

In [ ]:
def encode_review(review_input):
    """Convertit un avis brut en dictionnaire de tenseurs BERT."""
    if isinstance(review_input, bytes):
        review_text = review_input.decode("utf-8")
    elif hasattr(review_input, "numpy"):
        review_text = review_input.numpy().decode("utf-8")
    else:
        review_text = str(review_input)

    return tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,      # Ajoute [CLS] et [SEP]
        max_length=MAX_LENGTH,
        padding="max_length",         # Pad jusqu'à MAX_LENGTH
        truncation=True,              # Tronque si trop long
        return_attention_mask=True,   # Masque d'attention
        return_token_type_ids=True,   # IDs de segments
    )


def tf_encode(text, label):
    """Encapsule encode_review dans un tf.py_function pour le pipeline tf.data."""
    encoded = tf.py_function(
        func=lambda t: list(encode_review(t).values()),
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32]
    )
    return {
        "input_ids":      encoded[0],
        "attention_mask": encoded[1],
        "token_type_ids": encoded[2]
    }, label


def prepare_dataset(dataset):
    """Applique la tokenisation, le shuffle, le batching et le prefetch."""
    return (
        dataset
        .map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
        .shuffle(2000)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )


train_ds = prepare_dataset(ds_train)
test_ds  = prepare_dataset(ds_test)

print("Datasets préparés ✅")

---
## 🤖 Section 5 — Initialisation du modèle

Nous chargeons `TFBertForSequenceClassification` qui regroupe :
- L'**encodeur BERT** (110M paramètres pré-entraînés)
- Une **tête de classification** linéaire (2 classes : positif / négatif)

> 💡 Le taux d'apprentissage `2e-5` est très faible pour ne pas « oublier » les connaissances pré-entraînées (*catastrophic forgetting*).

In [ ]:
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn   = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics   = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()

---
## 🏋️ Section 6 — Entraînement et supervision

Sur un GPU T4 (Google Colab), **2 époques ≈ 15 minutes**.  

> 📈 Surveillez la précision de validation : si elle stagne ou régresse, c'est le signal pour arrêter (*early stopping*).

In [ ]:
EPOCHS = 2  # Augmentez à 3 si le temps le permet

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=test_ds,
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Courbe de précision
axes[0].plot(epochs_range, history.history["accuracy"],     label="Train Accuracy", marker="o")
axes[0].plot(epochs_range, history.history["val_accuracy"], label="Val Accuracy",   marker="o")
axes[0].set_title("Précision par époque")
axes[0].set_xlabel("Époque")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True)

# Courbe de perte
axes[1].plot(epochs_range, history.history["loss"],     label="Train Loss", marker="o")
axes[1].plot(epochs_range, history.history["val_loss"], label="Val Loss",   marker="o")
axes[1].set_title("Perte par époque")
axes[1].set_xlabel("Époque")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig("learning_curves.png", dpi=150)
plt.show()
print("Courbes sauvegardées dans learning_curves.png")

---
## ✅ Section 7 — Évaluation sur l'ensemble de test

Même si `model.fit` fournit déjà des métriques de validation, nous relançons l'évaluation sur le pipeline de test **intact** pour simuler une assurance qualité en production.

> ✅ Vérifiez si la précision dépasse le seuil de référence de **~0,90** attendu en classe.

In [ ]:
eval_metrics = model.evaluate(test_ds, verbose=1)

print("\n" + "="*40)
print(f"  Loss     : {eval_metrics[0]:.4f}")
print(f"  Accuracy : {eval_metrics[1]:.4f}  ({eval_metrics[1]*100:.2f}%)")
print("="*40)

seuil = 0.90
if eval_metrics[1] >= seuil:
    print(f"✅ Seuil de {seuil*100:.0f}% atteint — modèle prêt pour le pilote support.")
else:
    print(f"⚠️  Seuil de {seuil*100:.0f}% non atteint — envisagez une 3e époque ou un nettoyage des données.")

---
## 🧭 Section 8 — Assistant d'inférence réutilisable

Nous intégrons tout dans une fonction pour coller de vraies transcriptions et obtenir un score instantané.

> 🧭 Les **scores de confiance** sont essentiels pour décider s'il faut répondre automatiquement ou escalader vers un humain.

In [ ]:
import numpy as np

def predict_sentiment(text: str):
    """
    Prédit le sentiment d'un texte et retourne le label ainsi que la confiance.
    
    Args:
        text: Texte à analyser (avis client, commentaire, ticket, etc.)
    
    Returns:
        label      : str  — "Positive" ou "Negative"
        confidence : float — score de confiance entre 0 et 1
    """
    # 1. Tokenisation
    encoded = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors="tf"
    )

    # 2. Inférence (batch de taille 1)
    inputs = {
        "input_ids":      encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "token_type_ids": encoded["token_type_ids"]
    }
    logits = model(inputs, training=False).logits  # shape: (1, 2)

    # 3. Conversion en probabilités via softmax
    probs = tf.nn.softmax(logits, axis=-1).numpy()[0]  # shape: (2,)

    # 4. Label final
    label = "Positive" if np.argmax(probs) == 1 else "Negative"

    return label, float(probs.max())


# --- Test rapide ---
custom_sentence = "The onboarding emails were confusing, but the agent fixed everything politely."
label, confidence = predict_sentiment(custom_sentence)
print(f"Texte      : {custom_sentence}")
print(f"Prédiction : {label} (confiance={confidence:.3f})")

In [ ]:
# Exemples de tickets support réels
tickets_support = [
    "I've been waiting 3 weeks for a refund and no one is responding. This is unacceptable!",
    "The new interface is clean and intuitive. I found everything I needed in under a minute.",
    "Your app keeps crashing every time I try to upload a file. Very frustrating.",
    "Thank you for the quick resolution! The support team was incredibly helpful.",
    "I'm not sure how to interpret the billing statement. Can someone explain it?"
]

print(f"{'Ticket':<70} {'Label':<10} {'Confiance'}")
print("-" * 95)
for ticket in tickets_support:
    lbl, conf = predict_sentiment(ticket)
    flag = "🔴" if lbl == "Negative" else "🟢"
    print(f"{ticket[:68]:<70} {flag} {lbl:<8} {conf:.3f}")

---
## 💾 Section 9 — Sauvegarde du modèle

Sauvegardons le modèle fine-tuné pour une réutilisation future sans recalculer l'entraînement.

In [ ]:
# Sauvegarde du modèle et du tokenizer
SAVE_PATH = "./bert_sentiment_model"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"✅ Modèle sauvegardé dans : {SAVE_PATH}")
print("   Pour recharger :")
print("   model = TFBertForSequenceClassification.from_pretrained('./bert_sentiment_model')")
print("   tokenizer = BertTokenizer.from_pretrained('./bert_sentiment_model')")

---
## 🤔 Section 10 — Réflexion et prochaines étapes

### Ce que vous avez accompli
- ✅ Réutilisé un checkpoint public BERT pour atteindre >90% de précision avec un minimum de données
- ✅ Construit un pipeline complet : données → tokenisation → entraînement → évaluation → inférence
- ✅ Créé un assistant d'inférence réutilisable pour les tickets support réels

### Compétences transférables
| Domaine | Application |
|---------|-------------|
| RH | Analyse des enquêtes de satisfaction employés |
| Juridique | Classification de documents par tonalité |
| Produit | Analyse automatique des avis utilisateurs |
| Support | Escalade prioritaire des tickets négatifs |

### Prochaines étapes possibles
1. **Adaptation de domaine** : collecter les e-mails/tickets de votre entreprise pour un fine-tuning spécialisé
2. **Multilinguisme** : utiliser `distilbert-base-multilingual-cased` ou `xlm-roberta-base`
3. **Monitoring** : détecter la dérive des données (data drift) en production
4. **API REST** : exposer `predict_sentiment()` via FastAPI ou Flask

---
### ✏️ Questions de réflexion (répondre dans les cellules Markdown ci-dessous)

> 1. **Levier principal** : Quel levier (nettoyage des données, hyperparamètres, nombre d'époques) a le plus amélioré les résultats ?
> 2. **Garde-fous** : Où ajouteriez-vous des garde-fous avant de déployer ce signal de sentiment en production ?
> 3. **Parties prenantes** : Quels acteurs bénéficient le plus de cet outil (responsable support, chef de produit, responsable conformité) ?

### Réponse 1 — Levier principal

*Votre réponse ici...*

*(Indice : comparez l'impact de : passer de 1 à 2 époques, ajuster le learning rate de 5e-5 à 2e-5, ou réduire MAX_LENGTH de 512 à 256)*

### Réponse 2 — Garde-fous en production

*Votre réponse ici...*

*(Indice : pensez à : seuil de confiance minimum, revue humaine des cas ambigus, détection de langues non supportées, monitoring du biais)*

### Réponse 3 — Parties prenantes

*Votre réponse ici...*

*(Indice : chaque acteur a des besoins différents — le responsable support veut de l'escalade automatique, le chef de produit veut des tendances agrégées, le responsable conformité veut des logs d'audit)*